In [3]:
import pandas as pd

In [2]:
import os
print(os.getcwd())

c:\Users\admin\Desktop\10-academy\fraud-detection\notebooks


In [3]:
os.listdir("..")

['.git',
 '.github',
 '.gitignore',
 'data',
 'models',
 'notebooks',
 'README.md',
 'requirements.txt',
 'scripts',
 'src',
 'tests',
 'venv']

In [5]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\admin\Desktop\10-academy\fraud-detection\notebooks
['eda-creditcard.ipynb', 'eda-fraud-data.ipynb', 'feature-engineering.ipynb', 'modeling.ipynb', 'shap-explainability.ipynb', '__init__.py']


In [7]:
import os

print(os.listdir(".."))

['.git', '.github', '.gitignore', 'data', 'models', 'notebooks', 'README.md', 'requirements.txt', 'scripts', 'src', 'tests', 'venv']


In [8]:
import os

for root, dirs, files in os.walk(".."):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

..\data\raw\creditcard.csv
..\data\raw\Fraud_Data.csv
..\data\raw\IpAddress_to_Country.csv
..\venv\Lib\site-packages\matplotlib\mpl-data\sample_data\data_x_x2_x3.csv
..\venv\Lib\site-packages\matplotlib\mpl-data\sample_data\msft.csv
..\venv\Lib\site-packages\matplotlib\mpl-data\sample_data\Stocks.csv
..\venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-1.csv
..\venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-2.csv
..\venv\Lib\site-packages\numpy\random\tests\data\pcg64-testset-1.csv
..\venv\Lib\site-packages\numpy\random\tests\data\pcg64-testset-2.csv
..\venv\Lib\site-packages\numpy\random\tests\data\pcg64dxsm-testset-1.csv
..\venv\Lib\site-packages\numpy\random\tests\data\pcg64dxsm-testset-2.csv
..\venv\Lib\site-packages\numpy\random\tests\data\philox-testset-1.csv
..\venv\Lib\site-packages\numpy\random\tests\data\philox-testset-2.csv
..\venv\Lib\site-packages\numpy\random\tests\data\sfc64-testset-1.csv
..\venv\Lib\site-packages\numpy\random\tests\data\sfc6

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Fraud_Data.csv")
ip_df = pd.read_csv("../data/raw/IpAddress_to_Country.csv")

print(df.shape)
df.head()

(151112, 11)


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


In [11]:
df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])

df[['signup_time', 'purchase_time']].head()

,signup_time,purchase_time
0,2015-02-24 22:55:49,2015-04-18 02:47:11
1,2015-06-07 20:39:50,2015-06-08 01:38:54
2,2015-01-01 18:52:44,2015-01-01 18:52:45
3,2015-04-28 21:13:25,2015-05-04 13:54:50
4,2015-07-21 07:09:52,2015-09-09 18:40:53


# Feature 1 - Time Since Signup

In [12]:
df['time_since_signup'] = (
    df['purchase_time'] - df['signup_time']
).dt.total_seconds()

df['time_since_signup'].head()

0    4506682.0
1      17944.0
2          1.0
3     492085.0
4    4361461.0
Name: time_since_signup, dtype: float64

# Feature 2 - Hour of Day

In [13]:
df['hour_of_day'] = df['purchase_time'].dt.hour

df['hour_of_day'].head()

0     2
1     1
2    18
3    13
4    18
Name: hour_of_day, dtype: int32

# Feature 3 - Day of Week

In [14]:
df['day_of_week'] = df['purchase_time'].dt.dayofweek

df['day_of_week'].head()

0    5
1    0
2    3
3    0
4    2
Name: day_of_week, dtype: int32

# Feature 4 - Transaction Count

In [15]:
df['transaction_count'] = (
    df.groupby('user_id')['user_id']
      .transform('count')
)

df['transaction_count'].head()

0    1
1    1
2    1
3    1
4    1
Name: transaction_count, dtype: int64

#    Feature 5 - Device Frequency

In [17]:
df['device_frequency'] = (
    df.groupby('device_id')['device_id']
      .transform('count')
)

df['device_frequency'].head()

0     1
1     1
2    12
3     1
4     1
Name: device_frequency, dtype: int64

# Verify New Features

In [18]:
df[[
    'time_since_signup',
    'hour_of_day',
    'day_of_week',
    'transaction_count',
    'device_frequency'
]].head()

,time_since_signup,hour_of_day,day_of_week,transaction_count,device_frequency
0,4506682.0,2,5,1,1
1,17944.0,1,0,1,1
2,1.0,18,3,1,12
3,492085.0,13,0,1,1
4,4361461.0,18,2,1,1


# Prepare IP Address Merge

In [19]:
df['ip_address'] = df['ip_address'].astype(np.int64)

ip_df.head()

,lower_bound_ip_address,upper_bound_ip_address,country
0,16777216.0,16777471,Australia
1,16777472.0,16777727,China
2,16777728.0,16778239,China
3,16778240.0,16779263,Australia
4,16779264.0,16781311,China


# Sort Data

In [21]:
df = df.sort_values('ip_address')

ip_df = ip_df.sort_values('lower_bound_ip_address')

Let's find the exact location of your CSV files instead of guessing.

In a new notebook cell, run:

import os

print("Current folder:")
print(os.getcwd())

print("\nFiles here:")
print(os.listdir())

Then run:

import os

print(os.listdir(".."))

If that works, also run:

import os

for root, dirs, files in os.walk(".."):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

This will show something like:

..\data\Fraud_Data.csv
..\data\creditcard.csv
..\data\IpAddress_to_Country.csv

or

..\Fraud_Data.csv
..\creditcard.csv
..\IpAddress_to_Country.csv

or another path.

Copy and paste the output here. Once I see the actual file locations, I'll give you the exact pd.read_csv(...) commands and then guide you through the feature engineering notebook step by step.

import os

print(os.getcwd())
print(os.listdir())   i run this the output is this
c:\Users\admin\Desktop\10-academy\fraud-detection\notebooks
['eda-creditcard.ipynb', 'eda-fraud-data.ipynb', 'feature-engineering.ipynb', 'modeling.ipynb', 'shap-explainability.ipynb', '__init__.py']

Great. Your notebook is running from:

c:\Users\admin\Desktop\10-academy\fraud-detection\notebooks

Now we need to see where the CSV files are located.

Run this next:

import os

print(os.listdir(".."))

Then run:

import os

for root, dirs, files in os.walk(".."):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

The second command will list the exact paths of:

Fraud_Data.csv
creditcard.csv
IpAddress_to_Country.csv

For example, it might show:

..\data\Fraud_Data.csv
..\data\creditcard.csv
..\data\IpAddress_to_Country.csv

or

..\Fraud_Data.csv
..\creditcard.csv
..\IpAddress_to_Country.csv

Send me that output, and I'll tell you the exact code to use in feature-engineering.ipynb before we move on to creating:

time_since_signup
hour_of_day
day_of_week
transaction_count
device_frequency
IP-to-country merge
encoding
scaling
SMOTE

in the correct order.

..\data\raw\creditcard.csv
..\data\raw\Fraud_Data.csv
..\data\raw\IpAddress_to_Country.csv
..\venv\Lib\site-packages\matplotlib\mpl-data\sample_data\data_x_x2_x3.csv
..\venv\Lib\site-packages\matplotlib\mpl-data\sample_data\msft.csv
..\venv\Lib\site-packages\matplotlib\mpl-data\sample_data\Stocks.csv
..\venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-1.csv
..\venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-2.csv
..\venv\Lib\site-packages\numpy\random\tests\data\pcg64-testset-1.csv
..\venv\Lib\site-packages\numpy\random\tests\data\pcg64-testset-2.csv
..\venv\Lib\site-packages\numpy\random\tests\data\pcg64dxsm-testset-1.csv
..\venv\Lib\site-packages\numpy\random\tests\data\pcg64dxsm-testset-2.csv
..\venv\Lib\site-packages\numpy\random\tests\data\philox-testset-1.csv
..\venv\Lib\site-packages\numpy\random\tests\data\philox-testset-2.csv
..\venv\Lib\site-packages\numpy\random\tests\data\sfc64-testset-1.csv
..\venv\Lib\site-packages\numpy\random\tests\data\sfc64-testset-2.csv
..\venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arccos.csv
..\venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arccosh.csv
..\venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arcsin.csv
..\venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arcsinh.csv
..\venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arctan.csv
..\venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arctanh.csv
..\venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-cbrt.csv
..\venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-cos.csv
..\venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-cosh.csv
...
..\venv\Lib\site-packages\sklearn\datasets\data\linnerud_exercise.csv
..\venv\Lib\site-packages\sklearn\datasets\data\linnerud_physiological.csv
..\venv\Lib\site-packages\sklearn\datasets\data\wine_data.csv
..\venv\Lib\site-packages\tornado\test\csv_translations\fr_FR.csv
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...       this is the output

Perfect. Now we know exactly where your files are:

..\data\raw\Fraud_Data.csv
..\data\raw\creditcard.csv
..\data\raw\IpAddress_to_Country.csv

Since you're working on feature-engineering.ipynb for the fraud dataset, load the data like this:

import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Fraud_Data.csv")
ip_df = pd.read_csv("../data/raw/IpAddress_to_Country.csv")

print(df.shape)
df.head()
Next: Convert date columns

Run:

df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])

Check:

df[['signup_time', 'purchase_time']].head()
Feature 1: Time Since Signup

Run:

df['time_since_signup'] = (
    df['purchase_time'] - df['signup_time']
).dt.total_seconds()

Check:

df['time_since_signup'].head()
Feature 2: Hour of Day
df['hour_of_day'] = df['purchase_time'].dt.hour

df['hour_of_day'].head()
Feature 3: Day of Week
df['day_of_week'] = df['purchase_time'].dt.dayofweek

df['day_of_week'].head()
Feature 4: Transaction Count
df['transaction_count'] = (
    df.groupby('user_id')['user_id']
      .transform('count')
)

df['transaction_count'].head()
Feature 5: Device Frequency
df['device_frequency'] = (
    df.groupby('device_id')['device_id']
      .transform('count')
)

df['device_frequency'].head()
Verify the new columns

Run:

df[[
    'time_since_signup',
    'hour_of_day',
    'day_of_week',
    'transaction_count',
    'device_frequency'
]].head()

After these run successfully, send me the output of:

df.columns.tolist()

Then I'll guide you through the IP-to-country merge (Part 3), which is the most important part of Task 1.




user_id	signup_time	purchase_time	purchase_value	device_id	source	browser	sex	age	ip_address	class
0	22058	2015-02-24 22:55:49	2015-04-18 02:47:11	34	QVPSPJUOCKZAR	SEO	Chrome	M	39	7.327584e+08	0
1	333320	2015-06-07 20:39:50	2015-06-08 01:38:54	16	EOGFQPIZPYXFZ	Ads	Chrome	F	53	3.503114e+08	0
2	1359	2015-01-01 18:52:44	2015-01-01 18:52:45	15	YSSKYOSJHPPLJ	SEO	Opera	M	53	2.621474e+09	1
3	150084	2015-04-28 21:13:25	2015-05-04 13:54:50	44	ATGTXKYKUDUQN	SEO	Safari	M	41	3.840542e+09	0
4	221365	2015-07-21 07:09:52	2015-09-09 18:40:53	39	NAUITBZFJKHWW	Ads	Safari	M	45	4.155831e+08	0
after this output please guide me step by step of the next

Part 4: Feature Engineering

Notebook:

feature-engineering.ipynb
Feature 1

Time Since Signup

Required.

df['time_since_signup'] = (
    df['purchase_time'] - df['signup_time']
).dt.total_seconds()
Feature 2

Hour of Day

df['hour_of_day'] = df['purchase_time'].dt.hour
Feature 3

Day of Week

df['day_of_week'] = df['purchase_time'].dt.dayofweek
Feature 4

Transaction Frequency

Count transactions per user.

df['transaction_count'] = (
    df.groupby('user_id')['user_id']
      .transform('count')
)
Feature 5

Device Frequency

df['device_frequency'] = (
    df.groupby('device_id')['device_id']
      .transform('count')
)

Fraudsters often reuse devices.

Part 5: Data Transformation
One-Hot Encoding

Categorical columns:

source
browser
sex
country

Use:

pd.get_dummies()

or

OneHotEncoder()
Scaling

Numerical columns:

age
purchase_value
time_since_signup
transaction_count

Use:

StandardScaler()
Part 6: Handle Class Imbalance

Important instruction.

Apply ONLY to training data.

Split First
X_train, X_test, y_train, y_test
Then SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)
Show Before/After

Before:

y_train.value_counts()

After:

y_train_smote.value_counts()

Write justification:

SMOTE was chosen because it creates synthetic fraud examples while preserving valuable information from the majority class.

Great — your fraud dataset loaded correctly, and the datetime columns are already parsed properly. Now continue in feature-engineering.ipynb in this exact order.

Step 1: Create Time-Based Features

Feature 1: Time Since Signup

Feature 2: Hour of Day

Feature 3: Day of Week

Step 2: Create Frequency Features

Feature 4: Transaction Count per User

Feature 5: Device Frequency

Verify the new columns

Add a markdown cell:

Feature engineering created behavioral and temporal features that may help distinguish fraudulent from legitimate transactions. These include the time elapsed between signup and purchase, purchase hour, purchase day of week, transaction frequency per user, and device reuse frequency.

Step 3: Geolocation Integration (Important)

Before encoding, add the country information.

Convert IP addresses to integer

Sort both datasets

Merge using range lookup

Keep only rows where IP falls within the range

Now replace df with the merged result:

Quick check:

Step 4: One-Hot Encoding

Encode the categorical columns.

Add a markdown cell:

Categorical variables were one-hot encoded to convert them into numerical features suitable for machine learning models.

Step 5: Scaling

Scale only the numerical features that benefit from standardization.

Step 6: Train-Test Split and SMOTE

Prepare features and target

Split first (required)

Check class distribution before SMOTE

Apply SMOTE only on training data

Check class distribution after SMOTE

Add a markdown cell:

SMOTE was applied only to the training set to avoid data leakage. The technique generates synthetic minority-class examples, helping the model learn fraud patterns without discarding legitimate transactions.

Final Check Before Task 2

Run these checks:

If all cells run successfully, Task 1 is effectively complete and you can move on to modeling.ipynb for Task 2.

Next

Save, commit, and push

Then create a Pull Request

Title: Task 1: Data Analysis, Preprocessing, and Feature Engineering

please give me with code for each nothbook command(code)

Sure. Since you are currently in feature-engineering.ipynb, here is the exact code you can run cell by cell.

Cell 1: Load Libraries and Data
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Fraud_Data.csv")
ip_df = pd.read_csv("../data/raw/IpAddress_to_Country.csv")

df.head()
Cell 2: Convert Date Columns
df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])

df[['signup_time', 'purchase_time']].head()
Cell 3: Feature 1 - Time Since Signup
df['time_since_signup'] = (
    df['purchase_time'] - df['signup_time']
).dt.total_seconds()

df['time_since_signup'].head()
Cell 4: Feature 2 - Hour of Day
df['hour_of_day'] = df['purchase_time'].dt.hour

df['hour_of_day'].head()
Cell 5: Feature 3 - Day of Week
df['day_of_week'] = df['purchase_time'].dt.dayofweek

df['day_of_week'].head()
Cell 6: Feature 4 - Transaction Count
df['transaction_count'] = (
    df.groupby('user_id')['user_id']
      .transform('count')
)

df['transaction_count'].head()
Cell 7: Feature 5 - Device Frequency
df['device_frequency'] = (
    df.groupby('device_id')['device_id']
      .transform('count')
)

df['device_frequency'].head()
Cell 8: Verify New Features
df[[
    'time_since_signup',
    'hour_of_day',
    'day_of_week',
    'transaction_count',
    'device_frequency'
]].head()
Cell 9: Prepare IP Address Merge
df['ip_address'] = df['ip_address'].astype(np.int64)

ip_df.head()
Cell 10: Sort Data
df = df.sort_values('ip_address')

ip_df = ip_df.sort_values('lower_bound_ip_address')
Cell 11: Merge IP with Country

In [23]:
print(df['ip_address'].dtype)

print(ip_df['lower_bound_ip_address'].dtype)
print(ip_df['upper_bound_ip_address'].dtype)

int64
float64
int64


In [24]:
ip_df['lower_bound_ip_address'] = (
    ip_df['lower_bound_ip_address']
    .astype('int64')
)

In [25]:
print(df['ip_address'].dtype)
print(ip_df['lower_bound_ip_address'].dtype)
print(ip_df['upper_bound_ip_address'].dtype)

int64
int64
int64


In [26]:
df = df.sort_values('ip_address')

ip_df = ip_df.sort_values('lower_bound_ip_address')

# Merge IP with Country

In [27]:
merged_df = pd.merge_asof(
    df,
    ip_df,
    left_on='ip_address',
    right_on='lower_bound_ip_address',
    direction='backward'
)

merged_df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,time_since_signup,hour_of_day,day_of_week,transaction_count,device_frequency,lower_bound_ip_address,upper_bound_ip_address,country
0,62421,2015-02-16 00:17:05,2015-03-08 10:00:39,46,ZCLZTAJPCRAQX,Direct,Safari,M,36,52093,0,1763014.0,10,6,1,1,NaN,NaN,NaN
1,173212,2015-03-08 04:03:22,2015-03-20 17:23:45,33,YFGYOALADBHLT,Ads,IE,F,30,93447,0,1084823.0,17,4,1,1,NaN,NaN,NaN
2,242286,2015-05-17 16:45:54,2015-05-26 08:54:34,33,QZNVQTUITFTHH,Direct,FireFox,F,32,105818,0,749320.0,8,1,1,1,NaN,NaN,NaN
3,370003,2015-03-03 19:58:39,2015-05-28 21:09:13,33,PIBUQMBIELMMG,Ads,IE,M,40,117566,0,7434634.0,21,3,1,1,NaN,NaN,NaN
4,119824,2015-03-20 00:31:27,2015-04-05 07:31:46,55,WFIIFCPIOGMHT,Ads,Safari,M,38,131423,0,1407619.0,7,6,1,1,NaN,NaN,NaN


In [28]:
merged_df = merged_df[
    merged_df['ip_address']
    <= merged_df['upper_bound_ip_address']
]

merged_df.shape

(129146, 19)

In [29]:
merged_df[['country']].head()

,country
634,Australia
635,Thailand
636,China
637,China
638,Thailand


# Merge IP with Country

In [30]:
merged_df = pd.merge_asof(
    df,
    ip_df,
    left_on='ip_address',
    right_on='lower_bound_ip_address',
    direction='backward'
)

merged_df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,time_since_signup,hour_of_day,day_of_week,transaction_count,device_frequency,lower_bound_ip_address,upper_bound_ip_address,country
0,62421,2015-02-16 00:17:05,2015-03-08 10:00:39,46,ZCLZTAJPCRAQX,Direct,Safari,M,36,52093,0,1763014.0,10,6,1,1,NaN,NaN,NaN
1,173212,2015-03-08 04:03:22,2015-03-20 17:23:45,33,YFGYOALADBHLT,Ads,IE,F,30,93447,0,1084823.0,17,4,1,1,NaN,NaN,NaN
2,242286,2015-05-17 16:45:54,2015-05-26 08:54:34,33,QZNVQTUITFTHH,Direct,FireFox,F,32,105818,0,749320.0,8,1,1,1,NaN,NaN,NaN
3,370003,2015-03-03 19:58:39,2015-05-28 21:09:13,33,PIBUQMBIELMMG,Ads,IE,M,40,117566,0,7434634.0,21,3,1,1,NaN,NaN,NaN
4,119824,2015-03-20 00:31:27,2015-04-05 07:31:46,55,WFIIFCPIOGMHT,Ads,Safari,M,38,131423,0,1407619.0,7,6,1,1,NaN,NaN,NaN


# Keep Valid IP Ranges

In [31]:
merged_df = merged_df[
    merged_df['ip_address']
    <= merged_df['upper_bound_ip_address']
]

merged_df.shape

(129146, 19)

#  Check Country Column

In [32]:
merged_df[['country']].head()

,country
634,Australia
635,Thailand
636,China
637,China
638,Thailand


# Replace df with Merged Data

In [33]:
df = merged_df.copy()

df.shape

(129146, 19)

# Country Fraud Analysis

In [34]:
df.groupby('country')['class'].mean().sort_values(
    ascending=False
).head(10)

country
Turkmenistan             1.000000
Namibia                  0.434783
Sri Lanka                0.419355
Luxembourg               0.388889
Virgin Islands (U.S.)    0.333333
Ecuador                  0.264151
Tunisia                  0.262712
Peru                     0.260504
Bolivia                  0.245283
Kuwait                   0.233333
Name: class, dtype: float64

# One-Hot Encoding

In [35]:
df = pd.get_dummies(
    df,
    columns=[
        'source',
        'browser',
        'sex',
        'country'
    ],
    drop_first=True
)

df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,age,ip_address,class,time_since_signup,hour_of_day,...,country_United States,country_Uruguay,country_Uzbekistan,country_Vanuatu,country_Venezuela,country_Viet Nam,country_Virgin Islands (U.S.),country_Yemen,country_Zambia,country_Zimbabwe
634,247547,2015-06-28 03:00:34,2015-08-09 03:57:29,47,KIXYSVCHIPQBR,30,16778864,0,3632215.0,3,...,False,False,False,False,False,False,False,False,False,False
635,220737,2015-01-28 14:21:11,2015-02-11 20:28:28,15,PKYOWQKWGJNJI,34,16842045,0,1231637.0,20,...,False,False,False,False,False,False,False,False,False,False
636,390400,2015-03-19 20:49:09,2015-04-11 23:41:23,44,LVCSXLISZHVUO,29,16843656,0,1997534.0,23,...,False,False,False,False,False,False,False,False,False,False
637,69592,2015-02-24 06:11:57,2015-05-23 16:40:14,55,UHAUHNXXUADJE,30,16938732,0,7640897.0,16,...,False,False,False,False,False,False,False,False,False,False
638,174987,2015-07-07 12:58:11,2015-11-03 04:04:30,51,XPGPMOHIDRMGE,37,16971984,0,10249579.0,4,...,False,False,False,False,False,False,False,False,False,False


# Select Numerical Features

In [36]:
from sklearn.preprocessing import StandardScaler

numerical_cols = [
    'age',
    'purchase_value',
    'time_since_signup',
    'transaction_count'
]

# Scale Numerical Features

In [38]:
scaler = StandardScaler()

df[numerical_cols] = scaler.fit_transform(
    df[numerical_cols]
)

df[numerical_cols].head()

,age,purchase_value,time_since_signup,transaction_count
634,-0.363124,0.549607,-0.413800,0.0
635,0.101168,-1.197335,-1.180852,0.0
636,-0.479197,0.385831,-0.936126,0.0
637,-0.363124,0.986342,0.867086,0.0
638,0.449387,0.767974,1.700633,0.0


# Prepare X and y

In [39]:
X = df.drop(
    columns=[
        'class',
        'signup_time',
        'purchase_time',
        'device_id'
    ]
)

y = df['class']

print(X.shape)
print(y.shape)

(129146, 198)
(129146,)


# Train-Test Split

In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Before SMOTE

In [41]:
print("Before SMOTE:")
print(y_train.value_counts())

Before SMOTE:
class
0    93502
1     9814
Name: count, dtype: int64


# Apply SMOTE

In [42]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

# After SMOTE

In [43]:
print("After SMOTE:")
print(y_train_smote.value_counts())

After SMOTE:
class
0    93502
1    93502
Name: count, dtype: int64


# Save Processed Data

In [46]:
import os

os.makedirs("../data/processed", exist_ok=True)

print("Folder created successfully")

Folder created successfully


In [47]:
os.listdir("../data")

['processed', 'raw']

In [50]:
df.to_csv("../data/processed/fraud_processed.csv", index=False)

In [49]:
os.listdir("../data/processed")

['fraud_train_smote.csv']